# MGD YOLO26s → YOLO26n — Kaggle

Satu notebook standalone. Tambahkan Poltrada V3, aktifkan GPU/Internet, dan Secret `WANDB_API_KEY`. Run All menjalankan audit/numerical verification lalu training 100 epoch untuk setiap seed aktif. Seed 42 aktif; empat seed lainnya tersedia sebagai komentar. Tidak membutuhkan checkout repository atau MMDetection.

Metode mengikuti **Masked Generative Distillation**, Yang dkk., ECCV 2022: [paper](https://arxiv.org/abs/2205.01529), [kode deteksi penulis](https://github.com/yzd-v/MGD/blob/2c9da0b28625eb948db57afc02c824452c3910fe/det/mmdet/distillation/losses/mgd.py). Referensi lokal: `references/mgd`, commit dan SHA dikunci pada konfigurasi. Kode referensi berlisensi Apache-2.0; adaptasi PyTorch/YOLO26 ini mempertahankan atribusi penulis.

## 1. Konfigurasi penelitian

Mengikuti Native KD 10/CrossKD 11: teacher s v62 yang sama, student pretrained nano, 100 epoch, 640/batch16, AdamW lr0.001, AMP, seed yang dipasangkan. Default engine termasuk nbs64, scheduler dan augmentasi tetap berasal dari Ultralytics 8.4.155. Setiap seed dimulai dari student dan MGD baru. Bobot alpha dan mask ratio adalah default resmi detektor one-stage, bukan hasil tuning YOLO.

In [ ]:
EXPERIMENT_ID = "15_mgd_yolo26n_teacher_yolo26s"
METHOD = "mgd"
DATA = "/kaggle/input/datasets/ilhamrafiqin/poltrada-v3-yolo26-clean/data.yaml"
MODEL_WEIGHTS = "yolo26n.pt"
EPOCHS = 100
IMGSZ = 640
BATCH = 16
SEEDS_TO_RUN = [
    42,
    # 123,
    # 2024,
    # 7,
    # 3407,
]
SEED = SEEDS_TO_RUN[0] if SEEDS_TO_RUN else None
DEVICE = 0
WORKERS = 2
OPTIMIZER = "AdamW"
LR = 0.001
PATIENCE = 100
AMP = True
WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 3.0
CLOSE_MOSAIC = 10
EVAL_SPLIT = "val"
PROJECT = "/kaggle/working/results"
RUN_FULL_TRAINING = True
RUN_SMOKE_TEST = False  # True hanya untuk smoke 1 epoch tambahan, bukan hasil penelitian.
KEEP_LAST_LOCAL_EPOCHS = 1  # Versi setiap epoch tetap di W&B; snapshot gagal upload tidak dihapus.
MGD_ALPHA = 2e-5  # Default deteksi one-stage resmi; belum dituning untuk YOLO26.
MGD_MASK_RATIO = 0.65  # Proporsi posisi spasial yang dimatikan, bukan keep ratio.
EXPECTED_NC = 6
WANDB_ENABLED = True
WANDB_ENTITY = "ilhamrafiqin-brawijaya-university"
WANDB_PROJECT = "yolom26-kd-indotraffic"
WANDB_GROUP = "T26s_S26n"
TEACHER_RUN_ID = "q5kp9k9d"
TEACHER_ARTIFACT = "ilhamrafiqin-brawijaya-university/yolom26-kd-indotraffic/02_baseline_yolo26s-q5kp9k9d-checkpoints:v62"
TEACHER_SHA256 = "5bc3ae61f8da1c9f03916922de10146805c27c64a11e46edd766a00d0f6f18fd"
MGD_REFERENCE_COMMIT = "2c9da0b28625eb948db57afc02c824452c3910fe"
MGD_REFERENCE_SHA256 = "a91595b98b906363abff7374662638ccaa7cc1557b44ab61640e5d610b9649de"
RESUME_STATUS = "RESUME_NOT_VERIFIED"

## 2. Lingkungan dan autentikasi
PyTorch/CUDA bawaan Kaggle dipertahankan; tidak ada secret di kode notebook.

In [ ]:
import os
import subprocess
import sys
from importlib import metadata

for package, version in (("ultralytics", "8.4.155"), ("wandb", "0.28.1")):
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        print(f"[ENV] Installing {package}=={version}", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", f"{package}=={version}"])

import ast
import base64
import copy
import gc
import hashlib
import json
import math
import random
import shutil
import statistics
import time
import types
import typing
import urllib.request
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from ultralytics import YOLO, settings
from ultralytics import __version__ as ultralytics_version
from ultralytics.cfg import get_cfg
from ultralytics.data.utils import check_det_dataset
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.distill_model import DistillationModel, FeatureHook
from ultralytics.nn.modules.head import Detect
from ultralytics.nn.tasks import DetectionModel, load_checkpoint
from ultralytics.utils.torch_utils import (
    copy_attr,
    get_flops,
    init_seeds,
    strip_optimizer,
    unwrap_model,
)

import wandb

if ultralytics_version != "8.4.155" or wandb.__version__ != "0.28.1":
    raise RuntimeError("Restart kernel setelah instalasi: Ultralytics harus 8.4.155.")
settings.update({"wandb": False})
LAST_PHASE = {"value": "initialization"}


def status(stage: str, message: str) -> None:
    print(f"[{datetime.now().astimezone():%H:%M:%S}] [{stage}] {message}", flush=True)


def require_cuda() -> None:
    if not torch.cuda.is_available() or DEVICE != 0:
        raise RuntimeError("Aktifkan satu GPU CUDA Kaggle, DEVICE=0. Training penelitian memerlukan GPU.")


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def state_digest(model: nn.Module) -> str:
    """Hash parameters AND buffers, including teacher BatchNorm statistics."""
    digest = hashlib.sha256()
    for key, value in model.state_dict().items():
        digest.update(key.encode())
        digest.update(value.detach().cpu().contiguous().reshape(-1).view(torch.uint8).numpy().tobytes())
    return digest.hexdigest()


def git_commit() -> str | None:
    try:
        result = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=False)
        return result.stdout.strip() if result.returncode == 0 else None
    except OSError:
        return None


ENVIRONMENT_INFO = {
    "python": sys.version, "torch": torch.__version__, "cuda": torch.version.cuda,
    "ultralytics": metadata.version("ultralytics"), "wandb": metadata.version("wandb"),
    "git_commit": git_commit(), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
SESSION_DIR = Path(PROJECT) / EXPERIMENT_ID / datetime.now(UTC).strftime("%Y%m%dT%H%M%S_%fZ")
SESSION_DIR.mkdir(parents=True, exist_ok=False)
status("ENV", json.dumps(ENVIRONMENT_INFO))

In [ ]:
require_cuda()
if not WANDB_ENABLED:
    raise ValueError("W&B online wajib untuk audit dan perlindungan checkpoint per epoch.")
if not os.environ.get("WANDB_API_KEY"):
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login(key=os.environ["WANDB_API_KEY"])
if wandb.run is not None:
    raise RuntimeError("Ada run W&B aktif; selesaikan run itu atau restart kernel terlebih dahulu.")

## 3. Dataset dan teacher
Split, label, urutan kelas dan checkpoint teacher tidak diganti. Audit versi, producer, manifest, SHA-256, schema dan label dilakukan untuk setiap seed.

In [ ]:
def class_names(names: dict | list) -> list[str]:
    return ([str(names[key]) for key in sorted(names, key=lambda key: int(key))]
            if isinstance(names, dict) else [str(name) for name in names])


if EVAL_SPLIT != "val":
    raise ValueError("Pemilihan checkpoint dan evaluasi pilot hanya memakai val.")
if not Path(DATA).is_file():
    raise FileNotFoundError(f"Dataset Kaggle tidak ditemukan: {DATA}")
data_info = check_det_dataset(DATA, autodownload=False)
if int(data_info["nc"]) != EXPECTED_NC or not data_info.get("train") or not data_info.get("val"):
    raise ValueError("Poltrada V3 harus memiliki train/val dan enam kelas.")
DATASET_NAMES = class_names(data_info["names"])
status("DATA", f"{DATA} | kelas={DATASET_NAMES}")

In [ ]:
def download_teacher() -> tuple[str, dict[str, Any]]:
    """Use only the fixed s/v62 artifact, validating source, bytes, schema and labels."""
    api = wandb.Api()
    artifact = api.artifact(TEACHER_ARTIFACT, type="model")
    if artifact.version != "v62" or artifact.name.split(":")[0] != "02_baseline_yolo26s-q5kp9k9d-checkpoints":
        raise ValueError("Teacher harus koleksi YOLO26s q5kp9k9d:v62.")
    producer = artifact.logged_by()
    if producer is None or producer.id != TEACHER_RUN_ID or list(producer.path) != [WANDB_ENTITY, WANDB_PROJECT, TEACHER_RUN_ID]:
        raise ValueError("Producer artifact bukan eksperimen teacher yang ditetapkan.")
    entry = artifact.manifest.entries.get("best.pt")
    if entry is None:
        raise ValueError("Manifest artifact tidak berisi best.pt; cache lokal bukan fallback.")
    status("TEACHER", f"Mengunduh {TEACHER_ARTIFACT}")
    directory = Path(artifact.download(root="/kaggle/working/teacher_yolo26s_v62"))
    checkpoint = directory / "best.pt"
    if not checkpoint.is_file():
        raise FileNotFoundError(f"best.pt tidak tersedia: {checkpoint}")
    payload = checkpoint.read_bytes()
    if entry.size is None or len(payload) != entry.size:
        raise ValueError("Ukuran best.pt berbeda dari manifest.")
    if base64.b64encode(hashlib.md5(payload).digest()).decode() != entry.digest:
        raise ValueError("Checksum best.pt berbeda dari manifest.")
    digest = hashlib.sha256(payload).hexdigest()
    if digest != TEACHER_SHA256:
        raise ValueError("SHA-256 teacher berbeda dari Native/CrossKD.")
    del payload
    teacher = reference = None
    try:
        teacher = YOLO(str(checkpoint))
        family = Path(str(teacher.model.yaml.get("yaml_file", "")).replace("\\", "/")).stem
        if family not in {"yolo26", "yolo26s"} or teacher.model.yaml.get("scale") != "s":
            raise ValueError("Checkpoint bukan YOLO26s.")
        if int(teacher.model.model[-1].nc) != EXPECTED_NC or class_names(teacher.model.names) != DATASET_NAMES:
            raise ValueError("Jumlah atau urutan label teacher berbeda dari dataset.")
        reference = DetectionModel("yolo26s.yaml", nc=EXPECTED_NC, verbose=False)
        if {k: tuple(v.shape) for k, v in teacher.model.state_dict().items()} != {
            k: tuple(v.shape) for k, v in reference.state_dict().items()
        }:
            raise ValueError("Schema tensor checkpoint tidak cocok dengan YOLO26s enam kelas.")
        artifact_metadata = dict(artifact.metadata or {})
        producer_config = dict(producer.config or {})
        checkpoint_args = dict((teacher.ckpt or {}).get("train_args") or {})
        declarations = {}
        for source, values in (("artifact", artifact_metadata), ("producer", producer_config), ("checkpoint", checkpoint_args)):
            dataset = values.get("dataset", values.get("data"))
            model = values.get("model")
            if dataset and str(dataset).replace("\\", "/") != DATA.replace("\\", "/"):
                raise ValueError(f"Dataset teacher berbeda pada {source}: {dataset}")
            if model and Path(str(model).replace("\\", "/")).name != "yolo26s.pt":
                raise ValueError(f"Deklarasi arsitektur teacher berbeda pada {source}: {model}")
            declarations[source] = {"dataset": dataset, "model": model}
        if not producer_config.get("dataset") or not producer_config.get("model") or not checkpoint_args.get("data"):
            raise ValueError("Provenance dataset/model teacher belum lengkap.")
        if not artifact.digest or not isinstance(artifact_metadata.get("epoch"), int):
            raise ValueError("Digest atau metadata epoch artifact belum tersedia; v62 bukan epoch.")
        teacher.model.eval().requires_grad_(False)
        provenance = {
            "requested_artifact": TEACHER_ARTIFACT, "resolved_name": artifact.name,
            "version": artifact.version, "artifact_digest": artifact.digest, "manifest_digest": entry.digest,
            "best_pt_sha256": digest, "checkpoint_path": str(checkpoint),
            "producer_id": producer.id, "producer_path": list(producer.path),
            "artifact_metadata": artifact_metadata, "declarations": declarations,
            "architecture": "yolo26s", "tensor_schema_verified": True, "nc": EXPECTED_NC,
            "names": DATASET_NAMES, "metadata_complete": True,
            "verified_at_utc": datetime.now(UTC).isoformat(),
        }
        status("TEACHER", f"{checkpoint} | artifact={artifact.name} | epoch={artifact_metadata.get('epoch')}")
        return str(checkpoint), provenance
    finally:
        teacher = reference = None
        gc.collect()

## 4. Masked Generative Distillation

Paper hal.5–6 Eq.2–6: alignment 1×1 → spatial mask → Conv3×3/ReLU/Conv3×3 → rekontruksi seluruh fitur teacher. Hal.8 §4.2: semua feature map neck, alpha2e-5/lambda0.65 untuk one-stage. Hal.13 mendukung dua conv penuh; generator tidak diperkecil.

Kode deteksi membentuk mask B×1×H×W dengan `U > 1-lambda` sebagai posisi nol, tanpa rescaling. Paper memakai `R < lambda`: distribusinya setara, bukan mask identik untuk angka acak yang sama. Implementasi mengikuti threshold kode. Loss per level = alpha × SSE / actual_batch; ketiga level **dijumlahkan**, tidak dirata-rata atau dinormalisasi terhadap C/H/W. Pembagian batch mengikuti kode resmi, tidak dituliskan eksplisit pada Eq.5.

Adaptasi YOLO memakai P3/P4/P5 (stride8/16/32), alignment bias dan generator penuh setiap level. Generator/alignment/reduksi berjalan FP32; AMP detector tetap aktif. Tambahan sekitar 6.37M parameter hanya untuk training. Paper menggunakan SGD/24 epoch dan pewarisan bobot teacher pada struktur yang cocok; eksperimen ini mempertahankan AdamW/100 epoch dan `yolo26n.pt` agar sebanding dengan Native/CrossKD. Tidak ada pewarisan neck/head teacher ke student atau KD metode lain.

In [ ]:
import torch
from torch import nn
from ultralytics.nn.modules.head import Detect


def neck_taps(model: nn.Module) -> list[int]:
    """Discover every final neck feature supplied to the three-scale YOLO26 Detect."""
    heads = [layer for layer in model.model if isinstance(layer, Detect)]
    if len(heads) != 1 or not isinstance(heads[0].f, (list, tuple)) or len(heads[0].f) != 3:
        raise ValueError("Model harus memiliki satu Detect dengan tiga input neck.")
    head = heads[0]
    if [float(value) for value in head.stride] != [8.0, 16.0, 32.0]:
        raise ValueError("Urutan stride harus P3/P4/P5 = 8/16/32.")
    indices = list(head.f)
    if any(not isinstance(index, int) or not 0 <= index < head.i for index in indices):
        raise ValueError("Koneksi neck Detect tidak kompatibel.")
    return indices


def teacher_neck_forward(model: nn.Module, image: torch.Tensor, last_index: int) -> None:
    """Traverse the teacher backbone/neck once without executing its Detect head."""
    saved, value = [], image
    for layer in model.model[:last_index + 1]:
        if layer.f != -1:
            value = (saved[layer.f] if isinstance(layer.f, int)
                     else [value if index == -1 else saved[index] for index in layer.f])
        value = layer(value)
        saved.append(value if layer.i in model.save else None)

In [ ]:
import math
from pathlib import Path
from typing import Any

import torch
from torch import nn
from torch.nn import functional as F
from ultralytics.nn.distill_model import DistillationModel
from ultralytics.nn.tasks import load_checkpoint
from ultralytics.utils.torch_utils import copy_attr


class MGDFeatureLoss(nn.Module):
    """Detection MGD: biased alignment, spatial mask, full generator, SSE/B.

    Reference: yzd-v/MGD commit 2c9da0b28625eb948db57afc02c824452c3910fe,
    det/mmdet/distillation/losses/mgd.py (not the channel-mask classification variant).
    FP32 arithmetic and explicit teacher detach preserve its mathematical objective
    while avoiding AMP overflow and accidental teacher gradients in this adapter.
    """
    def __init__(self, student_channels, teacher_channels, name,
                 alpha_mgd=0.00002, lambda_mgd=0.65):
        super().__init__()
        if not math.isfinite(alpha_mgd) or alpha_mgd < 0:
            raise ValueError("MGD_ALPHA harus finite dan nonnegative.")
        if not math.isfinite(lambda_mgd) or not 0 <= lambda_mgd <= 1:
            raise ValueError("MGD_MASK_RATIO harus finite di antara 0 dan 1.")
        if student_channels < 1 or teacher_channels < 1:
            raise ValueError("Jumlah channel fitur MGD harus positif.")
        self.alpha_mgd = float(alpha_mgd)
        self.lambda_mgd = float(lambda_mgd)
        self.name = name
        self.align = (nn.Conv2d(student_channels, teacher_channels, kernel_size=1,
                                stride=1, padding=0, bias=True)
                      if student_channels != teacher_channels else None)
        self.generation = nn.Sequential(
            nn.Conv2d(teacher_channels, teacher_channels, kernel_size=3, padding=1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(teacher_channels, teacher_channels, kernel_size=3, padding=1, bias=True))
        self.last_mask_rate = 0.0
        self.last_raw_loss = torch.zeros(())
        self.last_aligned_shape = []
        self.last_generated_shape = []

    def forward(self, preds_S, preds_T):
        if preds_S.ndim != 4 or preds_T.ndim != 4 or preds_S.shape[0] != preds_T.shape[0]:
            raise ValueError("MGD memerlukan pasangan fitur BCHW dengan batch yang sama.")
        if preds_S.shape[-2:] != preds_T.shape[-2:]:
            raise ValueError("Grid spasial MGD harus sama; interpolasi tidak diaktifkan.")
        with torch.autocast(device_type=preds_S.device.type, enabled=False):
            student = preds_S.float()
            if self.align is not None:
                student = self.align(student)
            self.last_aligned_shape = list(student.shape)
            raw = self.get_dis_loss(student, preds_T.detach().float())
            return raw * self.alpha_mgd

    def get_dis_loss(self, preds_S, preds_T):
        if preds_S.ndim != 4 or preds_S.shape != preds_T.shape or preds_T.shape[0] < 1:
            raise ValueError("Fitur MGD harus identik BCHW setelah alignment dan batch tidak kosong.")
        with torch.autocast(device_type=preds_S.device.type, enabled=False):
            student, teacher = preds_S.float(), preds_T.detach().float()
            batch, _, height, width = teacher.shape
            # Keep the official detection implementation's CPU RNG and strict threshold.
            mat = torch.rand((batch, 1, height, width)).to(student.device)
            mat = torch.where(mat > 1 - self.lambda_mgd, 0, 1).to(student.device)
            masked_feature = torch.mul(student, mat)  # Never mutate student neck features.
            reconstructed = self.generation(masked_feature)
            raw = F.mse_loss(reconstructed, teacher, reduction="sum") / batch
        if not torch.isfinite(raw):
            raise RuntimeError("MGD reconstruction loss non-finite; proses dihentikan.")
        self.last_mask_rate = float((mat == 0).float().mean().detach())
        self.last_raw_loss = raw.detach()
        self.last_generated_shape = list(reconstructed.shape)
        return raw


class MGDModel(DistillationModel):
    """Native YOLO26 detection plus detection-MGD on all three final neck maps."""
    def __init__(self, teacher_model: str | Path | nn.Module, student_model: nn.Module):
        # Avoid allocating/calling Native KD's score-weighted projector or Detect hooks.
        nn.Module.__init__(self)
        if isinstance(teacher_model, (str, Path)):
            teacher_model = load_checkpoint(teacher_model)[0]
        device = next(student_model.parameters()).device
        self.teacher_model = teacher_model.to(device).float()
        self.student_model = student_model
        self._freeze_teacher()
        if class_names(teacher_model.names) != class_names(student_model.names):
            raise ValueError("Urutan label teacher dan student aktual berbeda.")
        self.feats_idx = neck_taps(student_model)
        if neck_taps(teacher_model) != self.feats_idx:
            raise ValueError("Koneksi neck teacher/student berbeda; diperlukan graph YOLO26 yang sama.")
        self._teacher_feats, self._student_feats = {}, {}
        self._teacher_hooks, self._student_hooks = [], []
        self._register_feature_hooks()
        self.last_actual_batch = 0
        self.last_feature_shapes = {}
        previous_training = student_model.training
        try:
            student_model.eval()
            with torch.no_grad():
                image = torch.zeros(2, student_model.yaml.get("channels", 3), 64, 64, device=device)
                teacher_neck_forward(teacher_model, image, max(self.feats_idx))
                student_model(image)
            projectors, layout = [], []
            for name, index in zip(("P3", "P4", "P5"), self.feats_idx):
                student, teacher = self._student_feats[index], self._teacher_feats[index]
                if student.shape[-2:] != teacher.shape[-2:]:
                    raise ValueError("Ukuran spasial neck berbeda; tidak ada interpolasi diam-diam.")
                cin, cout = student.shape[1], teacher.shape[1]
                projectors.append(MGDFeatureLoss(cin, cout, name,
                                                alpha_mgd=MGD_ALPHA, lambda_mgd=MGD_MASK_RATIO))
                layout.append({"level": name, "index": index, "student_channels": cin,
                               "teacher_channels": cout,
                               "projection": "identity" if cin == cout else "conv1x1-bias",
                               "generation": "conv3x3-relu-conv3x3"})
            self.projector = nn.ModuleList(projectors).to(device)
            self.feature_layout = layout
            copy_attr(self, student_model)
            self.dis = 1.0  # Engine compatibility only; no Native KD loss is invoked.
        except BaseException:
            self._remove_feature_hooks()
            raise
        finally:
            self._teacher_feats.clear()
            self._student_feats.clear()
            student_model.train(previous_training)
            self._freeze_teacher()

    def forward(self, value: torch.Tensor | dict, *args: Any, **kwargs: Any):
        if isinstance(value, dict):
            return self.loss(value, *args, **kwargs)
        try:
            return self.student_model.predict(value, *args, **kwargs)
        finally:
            self._teacher_feats.clear()
            self._student_feats.clear()

    def loss(self, batch: dict, preds: Any = None):
        self._teacher_feats.clear()
        self._student_feats.clear()
        try:
            if self.training:
                self._freeze_teacher()
                self.last_actual_batch = int(batch["img"].shape[0])
                self.last_feature_shapes = {}
                LAST_PHASE["value"] = "teacher_forward"
                with torch.no_grad():
                    teacher_neck_forward(self.teacher_model, batch["img"], max(self.feats_idx))
                LAST_PHASE["value"] = "student_forward"
                preds = self.student_model(batch["img"])
            LAST_PHASE["value"] = "detection_loss"
            detection, items = self.student_model.loss(batch, preds)
            parts, weighted_parts, mask_rates = {}, {}, {}
            if self.training:
                LAST_PHASE["value"] = "mgd_forward"
                for position, (level, index) in enumerate(zip(("P3", "P4", "P5"), self.feats_idx)):
                    module = self.projector[position]
                    student, teacher = self._student_feats[index], self._teacher_feats[index]
                    weighted_parts[level] = module(student, teacher)
                    parts[level] = module.last_raw_loss
                    mask_rates[level] = module.last_mask_rate
                    self.last_feature_shapes[level] = {
                        "student": list(student.shape), "teacher": list(teacher.shape),
                        "aligned": module.last_aligned_shape, "generated": module.last_generated_shape}
            else:
                parts = {level: detection.new_zeros(()) for level in ("P3", "P4", "P5")}
                weighted_parts = dict(parts)
                mask_rates = {level: 0.0 for level in parts}
            raw, weighted = sum(parts.values()), sum(weighted_parts.values())
            if not torch.isfinite(detection).all() or not torch.isfinite(weighted):
                raise RuntimeError("Detection/MGD loss non-finite; proses dihentikan.")
            batch_size = int(batch["img"].shape[0])
            per_image_detection = detection.detach().sum() / batch_size
            items.update({f"mgd_{level}": value.detach() for level, value in parts.items()})
            items.update({f"mgd_mask_{level}": detection.new_tensor(rate) for level, rate in mask_rates.items()})
            items.update(mgd_loss=raw.detach(), mgd_weighted_loss=weighted.detach(),
                         det_total_loss=per_image_detection,
                         total_loss=per_image_detection + weighted.detach(),
                         mgd_to_det_ratio=weighted.detach() / per_image_detection.abs().clamp_min(1e-12))
            LAST_PHASE["value"] = "backward"
            # Reference raw MGD is SSE/B; apply alpha and the native batch factor once each.
            return torch.cat((detection.reshape(-1), (weighted * batch_size).reshape(1))), items
        finally:
            self._teacher_feats.clear()
            self._student_feats.clear()

In [ ]:
def numerical_verification():
    """Small raw-SSE/batch oracle; official-source loss/gradient parity is tested offline."""
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(0)
        block = MGDFeatureLoss(2, 2, "oracle", alpha_mgd=0.1, lambda_mgd=0.0)
        with torch.no_grad():
            for layer in block.generation:
                if isinstance(layer, nn.Conv2d):
                    layer.weight.zero_()
                    layer.bias.zero_()
                    for channel in range(2):
                        layer.weight[channel, channel, 1, 1] = 1
        student = torch.ones(2, 2, 3, 3, requires_grad=True)
        teacher = torch.zeros_like(student, requires_grad=True)
        loss = block(student, teacher)
        torch.testing.assert_close(loss, torch.tensor(1.8))  # 0.1 * (2*2*3*3) / 2.
        loss.backward()
        torch.testing.assert_close(student.grad, torch.full_like(student, 0.1))
        if teacher.grad is not None:
            raise RuntimeError("Teacher target tidak boleh menerima gradien.")
        block.lambda_mgd = 1.0
        torch.testing.assert_close(block(student.detach(), teacher.detach()), torch.tensor(0.0))
    return {"status": "PASS CPU ORACLE", "sse_divided_by_batch": True,
            "reference_sha256": MGD_REFERENCE_SHA256, "teacher_gradient_isolated": True}

print(numerical_verification())

## 5. Trainer dan checkpoint

Detection criterion O2M/O2O, optimizer, accumulation, clipping, EMA dan GradScaler native tetap. Teacher frozen/eval; teacher Detect tidak dijalankan. MGD ditambahkan dengan faktor actual batch tepat sekali. Tidak ada resize, batch-halving, AMP fallback atau recovery NaN otomatis. Native KD tidak dipanggil.

Validasi/inferensi hanya student; tidak mengonsumsi RNG mask. Snapshot `best.pt`/`last.pt` berupa student-only; state archive mencatat student/MGD/EMA/optimizer/scheduler/scaler/RNG. **RESUME_NOT_VERIFIED**: archive tidak menjamin pemulihan persis engine/prefetch; resume=True ditolak.

In [ ]:
def protocol_args(directory: Path, *, diagnostic: bool = False) -> dict[str, Any]:
    """Same explicit training protocol as Native 05; other engine defaults remain pinned."""
    return dict(data=DATA, epochs=1 if diagnostic else EPOCHS, imgsz=IMGSZ,  # noqa: C408 -- Keep reference training kwargs easy to compare.
                batch=BATCH,
                optimizer=OPTIMIZER, lr0=LR, seed=SEED, device=DEVICE,
                workers=WORKERS, patience=PATIENCE, amp=AMP,
                weight_decay=WEIGHT_DECAY, warmup_epochs=WARMUP_EPOCHS,
                close_mosaic=CLOSE_MOSAIC, project=str(directory), name="training", exist_ok=False,
                plots=not diagnostic, pretrained=True, save=True,
                distill_model=str(TEACHER_CHECKPOINT), dis=1.0, compile=False, resume=False)


def audit_optimizer(model: MGDModel, optimizer: torch.optim.Optimizer) -> None:
    expected = {id(parameter) for parameter in model.parameters() if parameter.requires_grad}
    actual = [id(parameter) for group in optimizer.param_groups for parameter in group["params"]]
    if set(actual) != expected or len(actual) != len(set(actual)):
        raise RuntimeError("Optimizer tidak mencakup seluruh parameter student/projector tepat sekali.")
    if model.teacher_model.training or any(parameter.requires_grad for parameter in model.teacher_model.parameters()):
        raise RuntimeError("Teacher harus frozen/eval.")
    if set(actual) & {id(parameter) for parameter in model.teacher_model.parameters()}:
        raise RuntimeError("Teacher masuk optimizer.")


class MGDTrainer(DetectionTrainer):
    """Small integration layer; retains native criterion, optimizer math, EMA and GradScaler."""
    def __init__(self, *args: Any, **kwargs: Any):
        if (kwargs.get("overrides") or {}).get("resume"):
            raise ValueError("RESUME_NOT_VERIFIED: pemulihan engine/loader belum didukung oleh notebook ini.")
        super().__init__(*args, **kwargs)
        self._requested_batch = self.args.batch
        self.add_callback("on_train_batch_start", self._strict_batch_start)
        self.add_callback("on_train_batch_end", self._strict_batch_end)

    def set_model_attributes(self) -> None:
        super().set_model_attributes()
        LAST_PHASE["value"] = "initialization"
        if self.args.resume or self.args.compile or self.world_size > 1:
            raise ValueError("MGD pilot: satu GPU, compile=False, resume=False.")
        self.model = MGDModel(self.args.distill_model, self.model)

    def _build_train_pipeline(self) -> None:
        if self.batch_size != self._requested_batch or self.args.batch != self._requested_batch:
            raise RuntimeError("Batch berubah sesudah OOM; protokol MGD tidak boleh diubah otomatis.")
        super()._build_train_pipeline()

    def build_optimizer(self, model: nn.Module, *args: Any, **kwargs: Any):
        optimizer = super().build_optimizer(model, *args, **kwargs)
        for group in optimizer.param_groups:
            group["params"] = [parameter for parameter in group["params"] if parameter.requires_grad]
        audit_optimizer(unwrap_model(model), optimizer)
        return optimizer

    @staticmethod
    def _strict_batch_start(trainer: Any) -> None:
        trainer._oom_retries = 3  # Disable the pinned engine's first-epoch OOM batch-halving path.
        if trainer.batch_size != trainer.args.batch or trainer.args.imgsz != IMGSZ:
            raise RuntimeError("Batch/resolusi efektif berbeda dari protokol.")
        if bool(trainer.amp) != AMP:
            raise RuntimeError("AMP yang diminta tidak aktif; hentikan, jangan fallback diam-diam.")
        LAST_PHASE["value"] = "batch_prepare"

    @staticmethod
    def _strict_batch_end(trainer: Any) -> None:
        trainer._oom_retries = 0
        if not torch.isfinite(trainer.loss):
            raise RuntimeError("Loss training tidak finite.")
        model = unwrap_model(trainer.model)
        if any(parameter.grad is not None for parameter in model.teacher_model.parameters()):
            raise RuntimeError("Gradien bocor ke teacher.")
        if model._teacher_feats or model._student_feats:
            raise RuntimeError("Feature hooks masih menahan tensor setelah loss.")

    def optimizer_step(self) -> None:
        LAST_PHASE["value"] = "optimizer_step"
        # Keep the pinned engine's unscale/clip/step/update/EMA/zero_grad unchanged.
        return super().optimizer_step()

    def validate(self):
        LAST_PHASE["value"] = "validation"
        return super().validate()

    def _handle_nan_recovery(self, epoch: int) -> bool:
        if not torch.isfinite(self.loss) or (self.fitness is not None and not math.isfinite(float(self.fitness))):
            raise RuntimeError("Loss/fitness non-finite; otomatis reload checkpoint dinonaktifkan pada pilot MGD.")
        return False

    def train(self):
        try:
            return super().train()
        finally:
            model = unwrap_model(self.model)
            if isinstance(model, DistillationModel):
                model._remove_feature_hooks()

In [ ]:
def cpu_tree(value: Any) -> Any:
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: cpu_tree(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return type(value)(cpu_tree(item) for item in value)
    return copy.deepcopy(value)


def training_state(trainer: Any) -> dict[str, Any]:
    """Portable tensor/state archive, not an advertised Ultralytics resume checkpoint."""
    model, ema = unwrap_model(trainer.model), unwrap_model(trainer.ema.ema)
    generator = getattr(trainer.train_loader, "generator", None)
    return {
        "format": "mgd-training-state-v1", "resume_status": RESUME_STATUS,
        "resume_limitations": ["Engine local iteration/accumulation counters and worker prefetch state are not restored/tested"],
        "epoch_zero_based": int(trainer.epoch), "seed": SEED,
        "student": cpu_tree(model.student_model.state_dict()), "projector": cpu_tree(model.projector.state_dict()),
        "ema_student": cpu_tree(ema.student_model.state_dict()), "ema_projector": cpu_tree(ema.projector.state_dict()),
        "ema_updates": int(trainer.ema.updates), "optimizer": cpu_tree(trainer.optimizer.state_dict()),
        "scheduler": cpu_tree(trainer.scheduler.state_dict()), "scaler": cpu_tree(trainer.scaler.state_dict()),
        "pending_gradients": {name: cpu_tree(parameter.grad) for name, parameter in model.named_parameters()
                              if parameter.requires_grad and parameter.grad is not None},
        "criterion_updates": getattr(model.criterion, "updates", None),
        "best_fitness": float(trainer.best_fitness), "fitness": float(trainer.fitness),
        "stopper": cpu_tree(vars(trainer.stopper)), "args": vars(trainer.args).copy(),
        "accumulate": int(trainer.accumulate), "rng_python": random.getstate(),
        "rng_numpy": np.random.get_state(), "rng_torch": torch.get_rng_state(),
        "rng_cuda": [torch.cuda.get_rng_state(trainer.device)] if trainer.device.type == "cuda" else [],
        "loader_generator_state": generator.get_state() if generator is not None else None,
        "teacher_provenance": TEACHER_PROVENANCE, "mgd_alpha": MGD_ALPHA, "mgd_mask_ratio": MGD_MASK_RATIO,
        "feature_layout": model.feature_layout, "environment": ENVIRONMENT_INFO,
    }


def export_student(source: Path, destination: Path) -> str:
    """Strip into a new file; never mutate the checkpoint the engine is still training with."""
    if source.resolve() == destination.resolve():
        raise ValueError("Snapshot student harus berbeda dari checkpoint training asli.")
    destination.parent.mkdir(parents=True, exist_ok=True)
    checkpoint = strip_optimizer(source, str(destination), updates={"teacher_provenance": TEACHER_PROVENANCE,
                                                                  "method": METHOD, "resume_status": "INFERENCE_ONLY"})
    model = checkpoint.get("model")
    if not isinstance(model, nn.Module) or isinstance(model, DistillationModel) or hasattr(model, "projector"):
        raise RuntimeError("Export gagal: inference checkpoint masih memerlukan MGD wrapper/projector.")
    if model.yaml.get("scale") != "n":
        raise RuntimeError("Export harus berupa student YOLO26n.")
    for layer in model.modules():
        if any(isinstance(hook, FeatureHook) for hook in layer._forward_hooks.values()):
            raise RuntimeError("Student inference masih memiliki hook KD.")
    del checkpoint, model
    return file_sha256(destination)


def snapshot_epoch(trainer: Any, context: dict[str, Any]) -> dict[str, Any]:
    """Immutable per-epoch snapshots, including class-independent training state archives."""
    epoch = int(trainer.epoch) + 1
    status("CHECKPOINT", f"Epoch {epoch}: menyiapkan snapshot student dan training-state.")
    snapshot = context["directory"] / "epoch_snapshots" / f"epoch-{epoch:03d}"
    snapshot.mkdir(parents=True, exist_ok=False)
    export_student(Path(trainer.last), snapshot / "last.pt")
    export_student(Path(trainer.best), snapshot / "best.pt")
    torch.save(training_state(trainer), snapshot / "training_state_last.pt")
    is_best = float(trainer.fitness) == float(trainer.best_fitness)
    if is_best:
        context["best_epoch"] = epoch
        shutil.copy2(snapshot / "training_state_last.pt", snapshot / "training_state_best.pt")
    elif context.get("best_state_path"):
        shutil.copy2(context["best_state_path"], snapshot / "training_state_best.pt")
    else:
        raise RuntimeError("Tidak ada training-state untuk best epoch; snapshot tidak boleh diberi label best.")
    context["best_state_path"] = snapshot / "training_state_best.pt"
    for name in ("results.csv", "args.yaml"):
        path = Path(trainer.save_dir) / name
        if not path.is_file():
            raise FileNotFoundError(f"File epoch wajib tidak tersedia: {path}")
        shutil.copy2(path, snapshot / name)
    (snapshot / "teacher_provenance.json").write_text(json.dumps(TEACHER_PROVENANCE, indent=2), encoding="utf-8")
    manifest = {path.name: {"sha256": file_sha256(path), "bytes": path.stat().st_size}
                for path in snapshot.iterdir() if path.is_file()}
    (snapshot / "checkpoint_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return {"directory": snapshot, "epoch": epoch, "is_best": is_best, "manifest": manifest}

## 6. W&B dan antrean seed
Group T26s_S26n; run ID baru, resume=never. Upload checkpoint setiap epoch ditunggu. Default hanya snapshot epoch terbaru yang sukses diunggah dipertahankan lokal (termasuk best.pt dan state best). Versi setiap epoch tetap di W&B. Snapshot gagal/belum terkonfirmasi upload tidak dihapus; kegagalan menghentikan antrean. Hasil seed lama tidak diimpor otomatis.

In [ ]:
def prune_uploaded_snapshots(context):
    """Bound local disk use only AFTER upload confirmation; retain all failed/unknown snapshots."""
    base = (context["directory"] / "epoch_snapshots").resolve()
    uploaded = sorted(context["uploaded_epochs"])
    for epoch in uploaded[:-KEEP_LAST_LOCAL_EPOCHS]:
        candidate = base / f"epoch-{epoch:03d}"
        resolved = candidate.resolve()
        if resolved.parent != base or not resolved.is_relative_to(context["directory"].resolve()):
            raise RuntimeError("Snapshot cleanup path keluar dari direktori run.")
        confirmation = candidate / "upload_status.json"
        if candidate.is_dir() and confirmation.is_file():
            metadata = json.loads(confirmation.read_text())
            if metadata.get("status") == "PASS" and metadata.get("epoch") == epoch:
                shutil.rmtree(candidate)


def gradient_norm(parameters):
    values = [p.grad.detach().float().norm() for p in parameters if p.grad is not None]
    norm = float(torch.stack(values).norm()) if values else 0.0
    if not math.isfinite(norm) or norm <= 0:
        raise RuntimeError("Gradien komponen trainable harus finite dan nonzero.")
    return norm


def make_callbacks(context):
    seed, run = SEED, context["run"]

    def guard():
        if SEED != seed or context["run"] is not run:
            raise RuntimeError("Callback berasal dari seed/run lain.")

    def start(trainer):
        guard()
        model = unwrap_model(trainer.model)
        audit_optimizer(model, trainer.optimizer)
        if trainer.batch_size != BATCH or trainer.args.imgsz != IMGSZ or bool(trainer.amp) != AMP:
            raise RuntimeError("Batch/imgsz/AMP efektif berbeda dari protokol.")
        context.update(teacher_before=state_digest(model.teacher_model), feature_layout=model.feature_layout,
                       optimizer_steps=0, amp_skipped_steps=0, gradient=None, started=time.perf_counter())
        run.config.update({"effective_training_args": vars(trainer.args), "feature_taps": model.feature_layout,
                           "mgd_trainable_parameters": sum(p.numel() for p in model.projector.parameters()),
                           "dataset_yaml_sha256": file_sha256(Path(DATA)), "resume_status": RESUME_STATUS})
        if trainer.device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(trainer.device)

        def before_step(optimizer, *_):
            found_inf = getattr(optimizer, "found_inf", None)
            if found_inf is not None and float(found_inf) != 0:
                context["amp_skipped_steps"] += 1
                return
            context["gradient"] = {
                "student": gradient_norm(model.student_model.parameters()),
                "alignment": gradient_norm(p for block in model.projector if block.align is not None for p in block.align.parameters()),
                "generation": gradient_norm(p for block in model.projector for p in block.generation.parameters()),
            }
            context["optimizer_steps"] += 1
        context["optimizer_hook"] = trainer.optimizer.register_step_pre_hook(before_step)

    def batch_end(trainer):
        guard()
        context["batches"] += 1
        model = unwrap_model(trainer.model)
        if model.teacher_model.training or any(p.grad is not None for p in model.teacher_model.parameters()):
            raise RuntimeError("Teacher keluar dari eval atau menerima gradien.")
        if model._teacher_feats or model._student_feats:
            raise RuntimeError("Cache fitur belum dibersihkan setelah loss.")
        if not all(math.isfinite(float(v)) for v in trainer.loss_items.values()):
            raise RuntimeError("Loss/metrik MGD non-finite.")

    def save(trainer):
        guard()
        epoch = int(trainer.epoch) + 1
        if epoch in context["uploaded_epochs"]:
            return
        LAST_PHASE["value"] = "checkpoint"
        record = snapshot_epoch(trainer, context)
        artifact = wandb.Artifact(f"{EXPERIMENT_ID}-{run.id}-checkpoints", type="model",
            metadata={"epoch": epoch, "best_epoch_so_far": context["best_epoch"], "seed": seed,
                      "method": METHOD, "mgd_alpha": MGD_ALPHA, "mgd_mask_ratio": MGD_MASK_RATIO,
                      "teacher_provenance": TEACHER_PROVENANCE, "resume_status": RESUME_STATUS,
                      "diagnostic_only": context["diagnostic"]})
        for path in sorted(record["directory"].iterdir()):
            # Snapshots are immutable until .wait() completes; avoid another persistent cache copy.
            artifact.add_file(str(path), name=path.name, policy="immutable", skip_cache=True)
        try:
            run.log_artifact(artifact, aliases=["latest", f"epoch-{epoch:03d}"] + (["best"] if record["is_best"] else [])).wait()
        except BaseException:
            (record["directory"] / "upload_status.json").write_text(json.dumps({"status": "FAIL", "epoch": epoch}))
            raise
        (record["directory"] / "upload_status.json").write_text(json.dumps({"status": "PASS", "epoch": epoch}))
        context["uploaded_epochs"].add(epoch)
        if epoch not in context["logged_epochs"]:
            values = {f"train/{key}": float(value) for key, value in trainer.tloss.items()}
            values.update({key: float(value) for key, value in trainer.metrics.items()})
            values.update(trainer.lr)
            values.update(epoch=epoch, optimizer_steps=context["optimizer_steps"], amp_skipped_steps=context["amp_skipped_steps"],
                          elapsed_seconds=time.perf_counter() - context["started"], best_epoch=context["best_epoch"])
            if context["gradient"]:
                values.update({"gradient/" + key: value for key, value in context["gradient"].items()})
            if trainer.device.type == "cuda":
                values.update(peak_allocated_gib=torch.cuda.max_memory_allocated(trainer.device) / 2**30,
                              peak_reserved_gib=torch.cuda.max_memory_reserved(trainer.device) / 2**30)
            run.log(values, step=epoch)
            context["logged_epochs"].add(epoch)
        prune_uploaded_snapshots(context)

    return {"on_train_start": start, "on_train_batch_end": batch_end, "on_model_save": save}

In [ ]:
def evaluate_student(best: Path, directory: Path) -> dict[str, Any]:
    model = None
    try:
        model = YOLO(str(best))
        if isinstance(model.model, DistillationModel) or model.model.yaml.get("scale") != "n":
            raise RuntimeError("Evaluasi final memerlukan checkpoint YOLO26n student-only.")
        LAST_PHASE["value"] = "validation"
        metrics = model.val(data=DATA, split=EVAL_SPLIT, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                            plots=True, verbose=False, project=str(directory), name="student_validation")
        try:
            flops = float(get_flops(model.model, imgsz=IMGSZ))
        except (AttributeError, ImportError, RuntimeError, TypeError, ValueError):
            flops = None
        return {"map50_95": float(metrics.box.map), "map50": float(metrics.box.map50),
                "precision": float(metrics.box.mp), "recall": float(metrics.box.mr),
                "ap_per_class": {DATASET_NAMES[int(index)]: float(metrics.box.maps[int(index)])
                                 for index in metrics.box.ap_class_index}, "class_names": DATASET_NAMES,
                "parameters_student": sum(parameter.numel() for parameter in model.model.parameters()),
                "gflops_student": flops, "inference_ms_per_image": float(metrics.speed.get("inference", 0)),
                "eval_split": "val"}
    finally:
        model = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def validate_research_settings():
    # Boundary values remain available to the loss unit oracle, not the research gradient gate.
    if not math.isfinite(MGD_ALPHA) or MGD_ALPHA <= 0:
        raise ValueError("Training MGD memerlukan MGD_ALPHA positif dan finite.")
    if not math.isfinite(MGD_MASK_RATIO) or not 0 <= MGD_MASK_RATIO < 1:
        raise ValueError("Training MGD memerlukan 0 <= MGD_MASK_RATIO < 1 agar alignment menerima gradien.")
    if isinstance(KEEP_LAST_LOCAL_EPOCHS, bool) or not isinstance(KEEP_LAST_LOCAL_EPOCHS, int) or KEEP_LAST_LOCAL_EPOCHS < 1:
        raise ValueError("KEEP_LAST_LOCAL_EPOCHS harus integer positif.")


def validate_seed_list(seeds):
    if not isinstance(seeds, (list, tuple)) or not seeds:
        raise ValueError("SEEDS_TO_RUN harus berisi minimal satu seed.")
    if any(isinstance(seed, bool) or not isinstance(seed, int) or not 0 <= seed < 2**32 for seed in seeds):
        raise ValueError("Seed harus integer 0 sampai 2**32-1.")
    if len(seeds) != len(set(seeds)):
        raise ValueError("Seed duplikat tidak diperbolehkan.")
    return list(seeds)


def run_one_seed(seed, queue_dir, *, diagnostic=False):
    global SEED, TEACHER_CHECKPOINT, TEACHER_PROVENANCE
    validate_research_settings()
    require_cuda()
    if wandb.run is not None:
        raise RuntimeError("Selesaikan run W&B aktif sebelum menjalankan seed baru.")
    if EVAL_SPLIT != "val" or not 1 <= EPOCHS <= 100:
        raise ValueError("Gunakan val dan 1 sampai 100 epoch.")
    SEED = seed
    init_seeds(seed, deterministic=True)
    directory = Path(queue_dir) / (("smoke-" if diagnostic else "seed-") + str(seed))
    directory.mkdir(parents=True, exist_ok=False)
    context = {"directory": directory, "diagnostic": diagnostic, "batch": BATCH, "batches": 0,
               "best_epoch": None, "uploaded_epochs": set(), "logged_epochs": set()}
    run = student = trainer = None
    started = time.perf_counter()
    try:
        # Re-audit the same fixed teacher for every independent seed; never inherit prior student weights.
        TEACHER_CHECKPOINT, TEACHER_PROVENANCE = download_teacher()
        if file_sha256(Path(TEACHER_CHECKPOINT)) != TEACHER_SHA256:
            raise RuntimeError("SHA-256 teacher berbeda dari teacher Native/CrossKD.")
        name = f"kd_mgd_T26s_S26n_seed{seed}" + ("_smoke" if diagnostic else "")
        run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
            group=f"diagnostic-{WANDB_GROUP}" if diagnostic else WANDB_GROUP,
            name=name, id=wandb.util.generate_id(), resume="never", job_type="diagnostic" if diagnostic else "training",
            config={"experiment": EXPERIMENT_ID, "method": METHOD, "teacher": "yolo26s", "model": MODEL_WEIGHTS,
                    "seed": seed, "seeds_to_run": list(SEEDS_TO_RUN), "teacher_artifact": TEACHER_ARTIFACT,
                    "teacher_provenance": TEACHER_PROVENANCE, "dataset": DATA,
                    "mgd_alpha": MGD_ALPHA, "mgd_mask_ratio": MGD_MASK_RATIO,
                    "reference_commit": MGD_REFERENCE_COMMIT, "reference_sha256": MGD_REFERENCE_SHA256,
                    "environment": ENVIRONMENT_INFO, "resume_status": RESUME_STATUS, "diagnostic_only": diagnostic},
            tags=[METHOD, "teacher-s", "student-n", f"seed{seed}", "poltrada-v3", "diagnostic" if diagnostic else "research"],
            settings=wandb.Settings(init_timeout=90))
        context["run"] = run
        linked = run.use_artifact(TEACHER_ARTIFACT, type="model")
        if linked.version != "v62":
            raise RuntimeError("Run harus menggunakan teacher artifact v62.")
        # Reseed after audits/probes so their allocations cannot change student/module initialization.
        init_seeds(seed, deterministic=True)
        student = YOLO(MODEL_WEIGHTS)
        for event, callback in make_callbacks(context).items():
            student.add_callback(event, callback)
        student.train(trainer=MGDTrainer, **protocol_args(directory, diagnostic=diagnostic))
        trainer = student.trainer
        model = unwrap_model(trainer.model)
        if state_digest(model.teacher_model) != context["teacher_before"]:
            raise RuntimeError("Teacher parameters atau BN berubah selama training.")
        if not context["uploaded_epochs"] or context["optimizer_steps"] < 1:
            raise RuntimeError("Belum ada epoch terunggah atau optimizer update nyata.")
        snapshot = directory / "epoch_snapshots" / f"epoch-{max(context['uploaded_epochs']):03d}"
        best = snapshot / "best.pt"
        evaluation = evaluate_student(best, directory)
        report = {"status": "PASS", **evaluation, "seed": seed, "diagnostic_only": diagnostic,
                  "completed_epochs": len(context["uploaded_epochs"]), "requested_epochs": 1 if diagnostic else EPOCHS,
                  "best_epoch": context["best_epoch"],
                  "student_checkpoint": str(best), "seed_directory": str(directory), "wandb_run_id": run.id,
                  "optimizer_steps": context["optimizer_steps"], "amp_skipped_steps": context["amp_skipped_steps"],
                  "teacher_parameters_and_bn_unchanged": True, "resume_status": RESUME_STATUS,
                  "duration_seconds": time.perf_counter() - started}
        (directory / "metrics.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
        (directory / "config.json").write_text(json.dumps({"effective_args": vars(trainer.args),
            "feature_taps": context["feature_layout"], "teacher_provenance": TEACHER_PROVENANCE,
            "mgd_alpha": MGD_ALPHA, "mgd_mask_ratio": MGD_MASK_RATIO, "environment": ENVIRONMENT_INFO,
            "reference_sha256": MGD_REFERENCE_SHA256, "resume_status": RESUME_STATUS}, indent=2, default=str), encoding="utf-8")
        final = wandb.Artifact(f"{EXPERIMENT_ID}-{run.id}-results", type="results", metadata={"seed": seed})
        for filename in ("metrics.json", "config.json"):
            final.add_file(str(directory / filename), name=filename)
        run.log_artifact(final, aliases=["final"]).wait()
        run.summary.update({f"final/{key}": value for key, value in report.items() if isinstance(value, (str, bool, int, float))})
        run.finish()
        return report
    except BaseException as error:
        (directory / "failure.json").write_text(json.dumps({"status": "FAIL", "seed": seed,
            "phase": LAST_PHASE["value"], "error_type": type(error).__name__, "message": str(error),
            "protocol_changed_automatically": False}, indent=2), encoding="utf-8")
        if run is not None:
            try:
                run.finish(exit_code=1)
            except Exception:  # noqa: BLE001 -- Preserve the training/upload exception.
                status("W&B", "Penutupan run gagal; antrean berhenti dan file lokal dipertahankan.")
        raise
    finally:
        if context.get("optimizer_hook") is not None:
            context["optimizer_hook"].remove()
        if student is not None and hasattr(student, "trainer"):
            for loader in (getattr(student.trainer, "train_loader", None), getattr(student.trainer, "test_loader", None)):
                if hasattr(loader, "close"):
                    loader.close()
        student = trainer = model = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def run_seed_queue():
    validate_research_settings()
    seeds = validate_seed_list(SEEDS_TO_RUN)
    if not RUN_FULL_TRAINING and not RUN_SMOKE_TEST:
        status("TRAIN", "Training dan smoke dinonaktifkan.")
        return []
    require_cuda()
    if not WANDB_ENABLED:
        raise ValueError("Eksperimen ini memerlukan W&B online.")
    if wandb.run is not None:
        raise RuntimeError("Masih ada run W&B aktif.")
    queue_dir = Path(PROJECT) / EXPERIMENT_ID / ("multiseed_" + datetime.now(UTC).strftime("%Y%m%dT%H%M%S_%fZ"))
    queue_dir.mkdir(parents=True, exist_ok=False)
    progress = {"seeds_to_run": seeds, "completed": [], "failed_seed": None, "status": "running"}
    results = []
    progress_path = queue_dir / "multiseed_progress.json"
    progress_path.write_text(json.dumps(progress, indent=2))
    for seed in seeds:
        try:
            if RUN_SMOKE_TEST:
                run_one_seed(seed, queue_dir, diagnostic=True)
            if RUN_FULL_TRAINING:
                result = run_one_seed(seed, queue_dir)
                results.append(result)
                progress["completed"].append({"seed": seed, "wandb_run_id": result["wandb_run_id"],
                                              "seed_directory": result["seed_directory"]})
                (queue_dir / "multiseed_results.json").write_text(json.dumps(results, indent=2))
            progress_path.write_text(json.dumps(progress, indent=2))
        except BaseException:
            progress.update(status="failed", failed_seed=seed)
            progress_path.write_text(json.dumps(progress, indent=2))
            raise
    progress["status"] = "completed"
    progress_path.write_text(json.dumps(progress, indent=2))
    return results

## 7. Jalankan training
Default hanya seed42, 100 epoch. Membuka komentar seed berikutnya menambahkan run serial yang independen, bukan melanjutkan model seed sebelumnya. Smoke opsional memakai satu epoch penuh dan run terpisah.

In [ ]:
MULTISEED_RESULTS = run_seed_queue()
print(json.dumps(MULTISEED_RESULTS, indent=2))

## 8. Batas pengujian

Default MGD belum dianggap optimal untuk YOLO26. Laporan numerik/CPU sintetis menguji software, bukan mAP Poltrada, kapasitas GPU atau stabilitas 100 epoch. **GPU Kaggle: NOT TESTED** sampai ada run aktual. Checkpoint dan log setiap epoch memberi bukti progres, bukan jaminan resume sesi.